# OpenShorts Studio on Kaggle: long video → vertical shorts with Stable Diffusion b-roll

Runs [OpenShorts](https://github.com/mutonby/openshorts) (MIT) on a free Kaggle GPU session, fully local and free:

| Step | Runs on |
|---|---|
| Transcription (faster-whisper `large-v3-turbo`), scene detection, face-tracked 9:16 reframing, subtitles | GPU 0 |
| Picking the best moments (Ollama, `qwen2.5:7b`) | GPU 1 |
| **B-roll** (new): moments + prompts from Ollama, images from Stable Diffusion (SDXL 1.0), slow zoom, cut into the clip | GPU 1 |

Everything is served on port **7860** behind a password and published through your Cloudflare Tunnel:
the OpenShorts dashboard at `/`, the B-roll page at `/broll`, and OpenShorts' MCP server at `/mcp` for agents.

### Before you run
1. **Accelerator:** GPU T4 x2. **Internet:** on. Run All takes roughly 10–15 minutes the first time (installs + model downloads).
2. **A Cloudflare Tunnel** for this notebook with a public hostname pointing to `http://localhost:7860`.
3. **Kaggle secrets** (Add-ons → Secrets, then attach them to this notebook):
   - `OPENSHORTS_TUNNEL_TOKEN` (required)
   - `OPENSHORTS_UI_PASSWORD` (recommended; otherwise a random one is printed below). Browser login: any username + this password. Agents: `Authorization: Bearer <password>` or `X-Access-Token: <password>`.
   - `OPENSHORTS_YT_COOKIES` (optional): your YouTube cookies in Netscape format, if YouTube blocks downloads from Kaggle.

### Good to know
- Cloudflare's free plan caps one upload at **100 MB**. For longer videos, attach the file as a Kaggle dataset and give OpenShorts its path, or use a direct download link.
- Not included (they need paid services): AI actors, lip-sync, ElevenLabs voices, auto-posting. The Remotion renderer is not started.
- `/kaggle/working` is wiped when the session ends; download the clips you want to keep.
- B-roll uses SDXL 1.0 base (CreativeML OpenRAIL++, commercial use allowed). SDXL-Turbo is much faster but its licence is non-commercial; switch in the settings cell if that suits you.

In [ ]:
# 1. Settings (edit if you like)
LLM_MODEL = "qwen2.5:7b"              # Ollama model that picks moments and b-roll prompts
WHISPER_MODEL = "large-v3-turbo"      # OpenShorts transcription
SD_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"   # OpenRAIL++ (commercial use OK), ~20 s per image on a T4
SD_STEPS = 20
# Faster but NON-COMMERCIAL licence: SD_MODEL = "stabilityai/sdxl-turbo"; SD_STEPS = 4
PORT = 7860
OPENSHORTS_COMMIT = 'faff66de0b91277510587a1c778be433ff3d26a4'
REPO_DIR = "/kaggle/working/openshorts"
APP_DIR = "/kaggle/working/studio_app"
print("Settings saved.")

In [ ]:
# 2. Install everything (about 8-12 minutes). Re-running skips what is already done.
import os, subprocess, sys, glob, shutil, time

def sh(cmd, check=True, quiet=True):
    t = time.time()
    p = subprocess.run(cmd, shell=True, capture_output=quiet, text=True)
    if check and p.returncode != 0:
        print((p.stdout or "")[-3000:], (p.stderr or "")[-3000:])
        raise RuntimeError("failed: " + cmd[:120])
    return p

print("System packages..."); sh("apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y ffmpeg fontconfig fonts-liberation fonts-noto-color-emoji zstd > /dev/null")

if not os.path.exists("/kaggle/working/cloudflared"):
    print("cloudflared..."); sh("curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /kaggle/working/cloudflared && chmod +x /kaggle/working/cloudflared")

if not shutil.which("ollama"):
    print("Ollama..."); sh("curl -fsSL https://ollama.com/install.sh | sh")

NODE_DIR = "/opt/node20"
if not os.path.exists(NODE_DIR + "/bin/node"):
    print("Node.js 20..."); sh("mkdir -p %s && curl -sL https://nodejs.org/dist/v20.18.1/node-v20.18.1-linux-x64.tar.xz | tar -xJ -C %s --strip-components=1" % (NODE_DIR, NODE_DIR))
os.environ["PATH"] = NODE_DIR + "/bin:" + os.environ["PATH"]

if not os.path.isdir(REPO_DIR + "/.git"):
    print("OpenShorts source..."); sh("git clone -q https://github.com/mutonby/openshorts %s" % REPO_DIR)
sh("cd %s && git fetch -q origin %s 2>/dev/null; git checkout -q %s" % (REPO_DIR, OPENSHORTS_COMMIT, OPENSHORTS_COMMIT), check=False)
print("OpenShorts at", sh("cd %s && git rev-parse --short HEAD" % REPO_DIR).stdout.strip())

print("Python packages (keeping Kaggle's own PyTorch)...")
reqs = [l.strip() for l in open(REPO_DIR + "/requirements.txt") if l.strip() and not l.startswith("#")]
keep = [r for r in reqs if not r.split("==")[0].strip().lower() in ("torch", "torchvision")]
open("/tmp/req_kaggle.txt", "w").write("\n".join(keep) + "\n")
if sh("pip install -q -r /tmp/req_kaggle.txt", check=False).returncode != 0:
    print("  Pinned versions did not all install on this Python; retrying with relaxed pins.")
    open("/tmp/req_relaxed.txt", "w").write("\n".join(r.split("==")[0] for r in keep) + "\n")
    sh("pip install -q -r /tmp/req_relaxed.txt")
sh('pip install -q --upgrade --pre "yt-dlp[default]"')
sh('pip install -q diffusers accelerate "nvidia-cublas-cu12<13" "nvidia-cudnn-cu12>=9,<10"')

# CUDA libraries for faster-whisper (CTranslate2)
libdirs = sorted({os.path.dirname(p) for pat in ("nvidia/cublas/lib/*.so*", "nvidia/cudnn/lib/*.so*", "nvidia/cuda_runtime/lib/*.so*")
                  for sp in sys.path if sp.endswith("site-packages") for p in glob.glob(os.path.join(sp, pat))})
os.environ["LD_LIBRARY_PATH"] = ":".join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]).strip(":")

# Subtitle fonts used by OpenShorts
os.makedirs(os.path.expanduser("~/.fonts"), exist_ok=True)
for f in glob.glob(REPO_DIR + "/fonts/*.ttf"):
    shutil.copy(f, os.path.expanduser("~/.fonts/"))
os.makedirs(os.path.expanduser("~/.config/fontconfig/conf.d"), exist_ok=True)
if os.path.exists(REPO_DIR + "/fonts/openshorts-fontmap.conf"):
    shutil.copy(REPO_DIR + "/fonts/openshorts-fontmap.conf", os.path.expanduser("~/.config/fontconfig/conf.d/60-openshorts.conf"))
sh("fc-cache -f", check=False)

# Dashboard: the self-hosted UI blocks every job until an Upload-Post (auto-posting) key is set,
# even with a local LLM. Auto-posting is optional here, so only the LLM is required.
app_jsx = REPO_DIR + "/dashboard/src/App.jsx"
src = open(app_jsx).read()
old = "const keysMissing = !billingEnabled && (!geminiOk || !uploadPostKey);"
if old in src:
    open(app_jsx, "w").write(src.replace(old, "const keysMissing = !billingEnabled && !geminiOk;"))
    print("Patched dashboard: Upload-Post key is optional.")
elif "!billingEnabled && !geminiOk;" not in src:
    print("WARNING: dashboard key check changed upstream; the UI may ask for an Upload-Post key. Pin OPENSHORTS_COMMIT back if so.")

dist = REPO_DIR + "/dashboard/dist"
if not os.path.exists(dist + "/index.html"):
    print("Building the dashboard (npm)...")
    sh("cd %s/dashboard && npm ci --no-audit --no-fund --loglevel=error && npm run build" % REPO_DIR)
idx = open(dist + "/index.html").read()
if "id=\"broll-link\"" not in idx:
    link = ('<a id="broll-link" href="/broll/" style="position:fixed;right:16px;bottom:16px;z-index:9999;'
            'background:#e4572e;color:#fff;font:600 14px system-ui;padding:10px 14px;border-radius:999px;'
            'text-decoration:none;box-shadow:0 6px 18px rgba(0,0,0,.25)">+ B-roll</a>')
    open(dist + "/index.html", "w").write(idx.replace("</body>", link + "</body>"))

import torch
print("Install complete. GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

### 3. App files
These cells only write files.

In [ ]:
# Gateway on port 7860: password, OpenShorts dashboard + backend passthrough, /broll
# Writes gateway.py into APP_DIR. Safe to re-run.
import os
APP_DIR = "/kaggle/working/studio_app"
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'gateway.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Gateway on port 7860: the one thing the Cloudflare Tunnel exposes.

- Password-protects everything (browser: HTTP Basic, any username; agents and MCP
  clients: ``Authorization: Bearer <password>`` or ``X-Access-Token: <password>``).
- Serves the built OpenShorts dashboard (SPA fallback to index.html).
- Streams /api, /videos, /thumbnails, /gallery, /video, /mcp, /health to the
  OpenShorts backend on 127.0.0.1:8000 (uploads and range requests included).
- Mounts the B-roll page and API at /broll.

Standard library + starlette/httpx/uvicorn (already installed with OpenShorts).
"""
import base64
import hmac
import os
import socket
import threading

import httpx
import uvicorn
from starlette.applications import Starlette
from starlette.requests import Request
from starlette.responses import FileResponse, JSONResponse, PlainTextResponse, Response, StreamingResponse
from starlette.routing import Mount, Route

BACKEND = os.environ.get("OPENSHORTS_BACKEND", "http://127.0.0.1:8000")
PROXY_PREFIXES = ("/api", "/videos", "/thumbnails", "/gallery", "/video", "/mcp", "/health")
HOP = {"connection", "keep-alive", "proxy-authenticate", "proxy-authorization", "te", "trailers",
       "transfer-encoding", "upgrade", "host", "content-length"}
CFG = {"password": None, "dist": None}
_client = None


def _client_get():
    global _client
    if _client is None:
        _client = httpx.AsyncClient(base_url=BACKEND, timeout=httpx.Timeout(None, connect=10.0), follow_redirects=False)
    return _client


# ------------------------------------------------------------------ auth
def _token_ok(given):
    pw = CFG["password"]
    return bool(given) and hmac.compare_digest(given.encode(), pw.encode())


def authorized(request):
    """Returns (ok, strip_authorization_header)."""
    if not CFG["password"]:
        return True, False
    if _token_ok(request.headers.get("x-access-token", "")):
        return True, False
    h = request.headers.get("authorization", "")
    if h.startswith("Basic "):
        try:
            _, _, given = base64.b64decode(h[6:]).decode("utf-8").partition(":")
        except Exception:
            given = ""
        if _token_ok(given):
            return True, True
    if h.startswith("Bearer ") and _token_ok(h[7:].strip()):
        return True, True
    return False, False


def denied(request):
    if request.url.path.startswith(("/api", "/mcp", "/broll/api")):
        return JSONResponse({"error": "unauthorized: send Authorization: Bearer <password> or X-Access-Token"},
                            status_code=401, headers={"WWW-Authenticate": 'Basic realm="OpenShorts Studio"'})
    return Response("Login required", status_code=401, headers={"WWW-Authenticate": 'Basic realm="OpenShorts Studio"'})


class AuthMiddleware:
    def __init__(self, app):
        self.app = app

    async def __call__(self, scope, receive, send):
        if scope["type"] != "http":
            return await self.app(scope, receive, send)
        request = Request(scope)
        if request.url.path in ("/healthz",):
            return await self.app(scope, receive, send)
        ok, strip = authorized(request)
        if not ok:
            return await denied(request)(scope, receive, send)
        if strip:   # never forward our password to OpenShorts
            scope = dict(scope)
            scope["headers"] = [(k, v) for k, v in scope["headers"] if k != b"authorization"]
        await self.app(scope, receive, send)


# ------------------------------------------------------------------ proxy
async def proxy(request: Request):
    client = _client_get()
    headers = [(k, v) for k, v in request.headers.items() if k.lower() not in HOP]
    headers.append(("x-forwarded-proto", request.headers.get("x-forwarded-proto", request.url.scheme)))
    url = httpx.URL(path=request.url.path, query=request.url.query.encode())
    body = request.stream() if request.method not in ("GET", "HEAD", "OPTIONS") else None
    try:
        req = client.build_request(request.method, url, headers=headers, content=body)
        upstream = await client.send(req, stream=True)
    except httpx.HTTPError as e:
        return JSONResponse({"error": "OpenShorts backend is not reachable: %s" % e.__class__.__name__}, status_code=502)
    out_headers = {k: v for k, v in upstream.headers.items() if k.lower() not in HOP and k.lower() != "content-encoding"}

    async def body_iter():
        try:
            async for chunk in upstream.aiter_raw():
                yield chunk
        finally:
            await upstream.aclose()

    return StreamingResponse(body_iter(), status_code=upstream.status_code, headers=out_headers)


async def render_unavailable(request):
    return JSONResponse({"error": "The Remotion renderer is not running in this notebook."}, status_code=503)


async def healthz(request):
    return JSONResponse({"ok": True})


# ------------------------------------------------------------------ dashboard
async def dashboard(request: Request):
    path = request.url.path
    if path.startswith(PROXY_PREFIXES):
        return await proxy(request)
    dist = CFG["dist"]
    if not dist or not os.path.isdir(dist):
        return PlainTextResponse("Dashboard not built. Run the install cell, or use /broll.", status_code=503)
    rel = os.path.normpath(path.lstrip("/")) if path != "/" else "index.html"
    full = os.path.realpath(os.path.join(dist, rel))
    if not full.startswith(os.path.realpath(dist) + os.sep):
        return PlainTextResponse("Not found", status_code=404)
    if os.path.isdir(full):
        full = os.path.join(full, "index.html")
    if os.path.isfile(full):
        cache = "public, max-age=31536000, immutable" if "/assets/" in full else "no-cache"
        return FileResponse(full, headers={"Cache-Control": cache})
    return FileResponse(os.path.join(dist, "index.html"), headers={"Cache-Control": "no-cache"})


def build_app(broll_app=None):
    routes = [Route("/healthz", healthz)]
    if broll_app is not None:
        routes.append(Mount("/broll", app=broll_app))
    routes += [Route("/render/{rest:path}", render_unavailable, methods=["GET", "POST", "PUT", "DELETE"]),
               Route("/{rest:path}", dashboard, methods=["GET", "HEAD", "POST", "PUT", "PATCH", "DELETE", "OPTIONS"])]
    app = Starlette(routes=routes)
    app.add_middleware(AuthMiddleware)
    return app


# ------------------------------------------------------------------ server
SERVERS = []


def start(port=7860, password=None, dist=None, broll_app=None):
    """Serve on IPv4 and IPv6 (cloudflared may dial [::1]) in background threads."""
    stop()
    CFG.update(password=password or None, dist=dist)
    app = build_app(broll_app)
    for family, host in ((socket.AF_INET, "0.0.0.0"), (socket.AF_INET6, "::")):
        # Bind the sockets here: uvicorn's own "::" socket is dual-stack and collides with 0.0.0.0.
        try:
            sock = socket.socket(family, socket.SOCK_STREAM)
            sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            if family == socket.AF_INET6:
                sock.setsockopt(socket.IPPROTO_IPV6, socket.IPV6_V6ONLY, 1)
            sock.bind((host, port))
        except OSError as e:
            print("Could not listen on [%s]:%d -> %s" % (host, port, e))
            continue
        cfg = uvicorn.Config(app, log_level="warning", proxy_headers=True,
                             forwarded_allow_ips="*", timeout_keep_alive=30)
        server = uvicorn.Server(cfg)
        server.install_signal_handlers = lambda: None   # running inside a notebook thread
        t = threading.Thread(target=server.run, kwargs={"sockets": [sock]}, daemon=True)
        t.start()
        SERVERS.append((server, t, sock))
    if not SERVERS:
        raise RuntimeError("nothing is listening on port %d" % port)
    return app


def stop():
    while SERVERS:
        server, t, sock = SERVERS.pop()
        server.should_exit = True
        t.join(timeout=5)
        try:
            sock.close()
        except OSError:
            pass
''')
print("wrote", os.path.join(APP_DIR, 'gateway.py'))

In [ ]:
# B-roll engine: Ollama picks moments, Stable Diffusion draws them, FFmpeg cuts them in
# Writes broll.py into APP_DIR. Safe to re-run.
import os
APP_DIR = "/kaggle/working/studio_app"
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'broll.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""B-roll for OpenShorts clips, all local and free.

clip.mp4 -> transcribe (faster-whisper) -> pick moments + image prompts (Ollama)
         -> images (Stable Diffusion, default SDXL 1.0 base) -> slow zoom (FFmpeg zoompan)
         -> cut the b-roll into the clip, original audio kept -> clip_broll.mp4

Mounted by the gateway at /broll:
  GET  /broll/                 page
  GET  /broll/api/clips        clips OpenShorts has made
  POST /broll/api/upload       add your own clip (mp4)
  GET  /broll/media/<path>     clips, images and results
  POST /broll/api/jobs         {"clip": "<job>/<file.mp4>", "count": 3, "mode": "upper"|"full", "style": "..."}
  GET  /broll/api/jobs/<id>    progress and result
"""
import glob
import json
import os
import re
import shutil
import subprocess
import threading
import time
import traceback
import uuid

import httpx
from starlette.applications import Starlette
from starlette.requests import Request
from starlette.responses import FileResponse, HTMLResponse, JSONResponse
from starlette.routing import Route

HERE = os.path.dirname(os.path.abspath(__file__))
CFG = {
    "output_dir": "output",                       # OpenShorts output dir (served at /videos)
    "ollama": "http://127.0.0.1:11434/v1",
    "llm_model": "qwen2.5:7b",
    "sd_model": "stabilityai/stable-diffusion-xl-base-1.0",   # OpenRAIL++: commercial use allowed
    "sd_steps": 20,
    "sd_device": "cuda:1",
    "whisper_model": "small",
    "whisper_device": "cuda",
    "whisper_index": 1,
    "fps": 30,
    "unload_after": False,
}
JOBS = {}
QUEUE_LOCK = threading.Lock()
_models = {"sd": None, "whisper": None}
STYLE = "photorealistic, cinematic lighting, shallow depth of field, high detail, vertical composition, no text, no watermark"
NEGATIVE = "text, letters, watermark, logo, blurry, distorted, deformed hands, extra fingers"


# ------------------------------------------------------------------ helpers
def run(cmd):
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError("ffmpeg failed: " + (p.stderr or p.stdout)[-1500:])
    return p.stdout


def probe(path):
    out = run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
               "stream=width,height,r_frame_rate:format=duration", "-of", "json", path])
    d = json.loads(out)
    st, fmt = d["streams"][0], d["format"]
    num, _, den = st.get("r_frame_rate", "30/1").partition("/")
    fps = float(num) / float(den or 1) if float(den or 1) else 30.0
    has_audio = bool(json.loads(run(["ffprobe", "-v", "error", "-select_streams", "a", "-show_entries",
                                     "stream=index", "-of", "json", path])).get("streams"))
    return {"w": int(st["width"]), "h": int(st["height"]), "fps": round(fps, 3) or 30.0,
            "duration": float(fmt["duration"]), "audio": has_audio}


def safe_clip_path(rel):
    root = os.path.realpath(CFG["output_dir"])
    full = os.path.realpath(os.path.join(root, rel))
    if not full.startswith(root + os.sep) or not full.endswith(".mp4") or not os.path.isfile(full):
        raise ValueError("unknown clip")
    return full


def list_clips():
    root = CFG["output_dir"]
    out = []
    for job_dir in sorted(glob.glob(os.path.join(root, "*")), key=os.path.getmtime, reverse=True):
        if not os.path.isdir(job_dir):
            continue
        job = os.path.basename(job_dir)
        title, source = job, None
        metas = glob.glob(os.path.join(job_dir, "*_metadata.json"))
        if metas:
            title = os.path.basename(metas[0])[:-len("_metadata.json")]
            try:
                source = json.load(open(metas[0])).get("source_video")
            except Exception:
                pass
        for f in sorted(glob.glob(os.path.join(job_dir, "*.mp4"))):
            name = os.path.basename(f)
            if name == source or name.endswith("_broll.mp4") or name.startswith(("temp_", "tmp_", ".")):
                continue
            broll = f[:-4] + "_broll.mp4"
            out.append({"clip": "%s/%s" % (job, name), "job": job, "title": title, "name": name,
                        "url": "/broll/media/%s/%s" % (job, name), "mtime": os.path.getmtime(f),
                        "broll_url": "/broll/media/%s/%s" % (job, os.path.basename(broll)) if os.path.exists(broll) else None})
    return out


# ------------------------------------------------------------------ 1. transcribe
def transcribe(path):
    if _models["whisper"] is None:
        from faster_whisper import WhisperModel
        dev = CFG["whisper_device"]
        _models["whisper"] = WhisperModel(CFG["whisper_model"], device=dev, device_index=CFG["whisper_index"],
                                          compute_type="float16" if dev == "cuda" else "int8")
    segs, _ = _models["whisper"].transcribe(path, vad_filter=True)
    return [{"start": round(s.start, 2), "end": round(s.end, 2), "text": s.text.strip()} for s in segs if s.text.strip()]


# ------------------------------------------------------------------ 2. plan with Ollama
PLAN_PROMPT = """You are a short-form video editor. Pick moments in this vertical clip where a b-roll image would help the viewer picture what is being said.

Clip length: {duration:.1f} seconds.
Transcript (start-end seconds: text):
{lines}

Return JSON only, in this exact shape:
{{"inserts": [{{"start": 4.2, "duration": 2.5, "prompt": "a detailed visual description for an image generator"}}]}}

Rules:
- Exactly {count} inserts, in time order, at least 1 second apart.
- Each duration between 1.5 and 3.5 seconds.
- Nothing before {lead:.1f}s or after {tail:.1f}s.
- Prompts describe a concrete scene that matches what is said at that moment. {style_hint}
- No text, captions, logos, brand names or real people's names in prompts.
"""


def plan(segments, duration, count, style):
    lead, tail = min(1.5, duration * .1), max(0.0, duration - 1.0)
    lines = "\n".join("%.1f-%.1f: %s" % (s["start"], s["end"], s["text"]) for s in segments) or "(no speech)"
    prompt = PLAN_PROMPT.format(duration=duration, lines=lines[:12000], count=count, lead=lead, tail=tail,
                                style_hint=("Visual style: " + style + ".") if style else "")
    inserts = []
    try:
        r = httpx.post(CFG["ollama"] + "/chat/completions", timeout=300, json={
            "model": CFG["llm_model"], "temperature": 0.4, "response_format": {"type": "json_object"},
            "messages": [{"role": "user", "content": prompt}]})
        r.raise_for_status()
        text = r.json()["choices"][0]["message"]["content"]
        m = re.search(r"\{.*\}", text, re.S)
        inserts = json.loads(m.group(0) if m else text).get("inserts", [])
    except Exception as e:
        print("[broll] LLM plan failed, using even spacing:", e)
    return clean_plan(inserts, segments, duration, count, lead, tail)


def clean_plan(inserts, segments, duration, count, lead, tail):
    out, last_end = [], -1e9
    for it in sorted((i for i in inserts if isinstance(i, dict)), key=lambda i: float(i.get("start", 0) or 0)):
        try:
            s, d = float(it["start"]), float(it.get("duration", 2.5))
        except (KeyError, TypeError, ValueError):
            continue
        d = min(3.5, max(1.5, d))
        s = max(lead, s)
        if s < last_end + 1.0 <= s + 1.5:      # too close to the previous insert: nudge it later a little
            s = last_end + 1.0
        if s + d > tail:
            s = tail - d
        p = str(it.get("prompt") or "").strip()
        if s < lead or not p or s < last_end + 1.0 - 1e-6:
            continue
        out.append({"start": round(s, 2), "duration": round(d, 2), "prompt": p[:400]})
        last_end = s + d
        if len(out) == count:
            break
    if not out and tail - lead >= 2:            # fallback: evenly spaced, prompt = what is said there
        n = max(1, min(count, int((tail - lead) // 4)))
        step = (tail - lead) / n
        for k in range(n):
            s = lead + k * step + max(0, (step - 2.5) / 2)
            said = " ".join(x["text"] for x in segments if x["end"] > s and x["start"] < s + 2.5) or "the topic of the video"
            out.append({"start": round(s, 2), "duration": 2.5, "prompt": "a scene showing " + said[:200]})
    return out


# ------------------------------------------------------------------ 3. images with Stable Diffusion
def image_size(info, mode):
    """Target aspect at the model's native size (~1 MP for SDXL, ~0.5 MP for Turbo/SD 1.x), multiples of 64."""
    w, h = info["w"], info["h"] if mode == "full" else int(info["h"] * 0.6)
    ratio = w / h
    m = CFG["sd_model"].lower()
    area = 768 * 1344 if ("xl" in m and "turbo" not in m) else 512 * 896
    gh = int(round((area / ratio) ** 0.5 / 64)) * 64
    gw = int(round(gh * ratio / 64)) * 64
    return max(256, min(1536, gw)), max(256, min(1536, gh))


def generate_image(prompt, style, size, out_path, seed):
    if _models["sd"] is None:
        import torch
        from diffusers import AutoPipelineForText2Image
        try:
            pipe = AutoPipelineForText2Image.from_pretrained(CFG["sd_model"], torch_dtype=torch.float16, variant="fp16")
        except Exception:                      # model without an fp16 variant
            pipe = AutoPipelineForText2Image.from_pretrained(CFG["sd_model"], torch_dtype=torch.float16)
        pipe = pipe.to(CFG["sd_device"])
        if hasattr(pipe, "enable_vae_tiling"):  # keeps the VAE decode within the T4's memory next to Ollama
            pipe.enable_vae_tiling()
        _models["sd"] = pipe
    import torch
    pipe = _models["sd"]
    g = torch.Generator(device=CFG["sd_device"]).manual_seed(seed)
    full_prompt = ", ".join(x for x in (prompt, style, STYLE) if x)
    turbo = "turbo" in CFG["sd_model"].lower()
    img = pipe(prompt=full_prompt, negative_prompt=None if turbo else NEGATIVE, width=size[0], height=size[1],
               num_inference_steps=CFG["sd_steps"], guidance_scale=0.0 if turbo else 6.0, generator=g).images[0]
    img.save(out_path)
    return out_path


# ------------------------------------------------------------------ 4. slow zoom (Ken Burns)
def ken_burns(image, out_path, w, h, duration, fps, k):
    frames = max(2, int(round(duration * fps)))
    zoom_in = k % 2 == 0
    z = "min(1+0.12*on/%d,1.12)" % frames if zoom_in else "max(1.12-0.12*on/%d,1)" % frames
    vf = ("scale=%d:%d:force_original_aspect_ratio=increase,crop=%d:%d,"
          "zoompan=z='%s':x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':d=%d:s=%dx%d:fps=%s,format=yuv420p"
          % (w * 2, h * 2, w * 2, h * 2, z, frames, w, h, fps))
    run(["ffmpeg", "-y", "-v", "error", "-loop", "1", "-i", image, "-vf", vf, "-frames:v", str(frames),
         "-c:v", "libx264", "-preset", "veryfast", "-crf", "18", out_path])
    return out_path


# ------------------------------------------------------------------ 5. cut into the clip
def composite(clip, inserts, info, mode, out_path):
    """Overlay each zoom clip during its window; 'upper' keeps the bottom 40% (burned-in captions) visible."""
    w, h = info["w"], info["h"]
    bh = h if mode == "full" else int(h * 0.6) // 2 * 2
    args = ["ffmpeg", "-y", "-v", "error", "-i", clip]
    for it in inserts:
        args += ["-i", it["video"]]
    chain, last = [], "0:v"
    for i, it in enumerate(inserts, 1):
        s, d = it["start"], it["duration"]
        f = min(0.25, d / 4)
        chain.append("[%d:v]scale=%d:%d,format=yuva420p,fade=t=in:st=0:d=%.2f:alpha=1,fade=t=out:st=%.2f:d=%.2f:alpha=1,"
                     "setpts=PTS-STARTPTS+%.3f/TB[b%d]" % (i, w, bh, f, d - f, f, s, i))
        chain.append("[%s][b%d]overlay=x=0:y=0:eof_action=pass:enable='between(t,%.3f,%.3f)'[v%d]" % (last, i, s, s + d, i))
        last = "v%d" % i
    args += ["-filter_complex", ";".join(chain), "-map", "[%s]" % last]
    if info["audio"]:
        args += ["-map", "0:a", "-c:a", "copy"]
    args += ["-c:v", "libx264", "-preset", "veryfast", "-crf", "20", "-pix_fmt", "yuv420p", "-movflags", "+faststart", out_path]
    run(args)
    return out_path


# ------------------------------------------------------------------ job runner
def _set(job, **kw):
    job.update(kw)
    job["updated"] = time.time()


def run_job(job):
    try:
        with QUEUE_LOCK:                      # one job at a time: the GPU is shared
            clip = safe_clip_path(job["clip"])
            info = probe(clip)
            work = os.path.join(os.path.dirname(clip), "broll_" + os.path.basename(clip)[:-4])
            os.makedirs(work, exist_ok=True)
            _set(job, state="running", step="Transcribing the clip", progress=0.05, info=info)
            segments = transcribe(clip)
            _set(job, step="Choosing moments with %s" % CFG["llm_model"], progress=0.2, transcript=segments)
            inserts = plan(segments, info["duration"], job["count"], job["style"])
            if not inserts:
                raise RuntimeError("clip is too short for b-roll")
            _set(job, inserts=inserts, progress=0.3)
            size = image_size(info, job["mode"])
            bh = info["h"] if job["mode"] == "full" else int(info["h"] * 0.6) // 2 * 2
            for k, it in enumerate(inserts):
                _set(job, step="Drawing image %d of %d" % (k + 1, len(inserts)), progress=0.3 + 0.5 * k / len(inserts))
                img = generate_image(it["prompt"], job["style"], size, os.path.join(work, "img_%d.png" % k), seed=job["seed"] + k)
                it["image_url"] = "/broll/media/%s/%s/img_%d.png" % (os.path.basename(os.path.dirname(clip)), os.path.basename(work), k)
                it["video"] = ken_burns(img, os.path.join(work, "kb_%d.mp4" % k), info["w"], bh, it["duration"], info["fps"], k)
                _set(job, inserts=inserts)
            _set(job, step="Cutting b-roll into the clip", progress=0.85)
            out = clip[:-4] + "_broll.mp4"
            composite(clip, inserts, info, job["mode"], out)
            rel = os.path.relpath(out, CFG["output_dir"]).replace(os.sep, "/")
            _set(job, state="done", step="Done", progress=1.0, result_url="/broll/media/" + rel)
    except Exception as e:
        traceback.print_exc()
        _set(job, state="error", step="Failed", error=str(e)[-1500:])
    finally:
        if CFG.get("unload_after"):            # single-GPU sessions: give the VRAM back to OpenShorts/Ollama
            unload_models()


def create_job(clip, count=3, mode="upper", style="", seed=None):
    safe_clip_path(clip)
    if mode not in ("upper", "full"):
        raise ValueError("mode must be 'upper' or 'full'")
    count = max(1, min(6, int(count)))
    job = {"id": uuid.uuid4().hex[:12], "clip": clip, "count": count, "mode": mode, "style": str(style or "")[:200],
           "seed": int(seed) if seed is not None else int(time.time()) % 100000, "state": "queued",
           "step": "Waiting for the GPU", "progress": 0.0, "created": time.time(), "updated": time.time()}
    JOBS[job["id"]] = job
    threading.Thread(target=run_job, args=(job,), daemon=True).start()
    return job


def public(job):
    j = {k: v for k, v in job.items() if k not in ("transcript",)}
    j["inserts"] = [{k: v for k, v in it.items() if k != "video"} for it in job.get("inserts", [])]
    return j


# ------------------------------------------------------------------ HTTP
async def page(request):
    return FileResponse(os.path.join(HERE, "broll_ui.html"))


async def media(request):
    root = os.path.realpath(CFG["output_dir"])
    full = os.path.realpath(os.path.join(root, request.path_params["path"]))
    if not full.startswith(root + os.sep) or not os.path.isfile(full):
        return JSONResponse({"error": "not found"}, status_code=404)
    return FileResponse(full)


async def api_clips(request):
    return JSONResponse({"clips": list_clips(), "model": CFG["llm_model"], "sd_model": CFG["sd_model"]})


async def api_jobs(request: Request):
    if request.method == "GET":
        return JSONResponse({"jobs": [public(j) for j in sorted(JOBS.values(), key=lambda j: -j["created"])][:30]})
    try:
        b = await request.json()
        job = create_job(b.get("clip", ""), b.get("count", 3), b.get("mode", "upper"), b.get("style", ""), b.get("seed"))
    except (ValueError, TypeError) as e:
        return JSONResponse({"error": str(e)}, status_code=400)
    return JSONResponse(public(job))


async def api_job(request):
    job = JOBS.get(request.path_params["jid"])
    if not job:
        return JSONResponse({"error": "unknown job"}, status_code=404)
    return JSONResponse(public(job))


async def api_upload(request: Request):
    """Raw mp4 body (Cloudflare's free plan caps a request at 100 MB)."""
    name = re.sub(r"[^A-Za-z0-9_.-]+", "-", request.query_params.get("name", "clip.mp4"))[:80] or "clip.mp4"
    if not name.lower().endswith(".mp4"):
        name += ".mp4"
    d = os.path.join(CFG["output_dir"], "uploads-broll")
    os.makedirs(d, exist_ok=True)
    path = os.path.join(d, "%s_%s" % (uuid.uuid4().hex[:6], name))
    size = 0
    with open(path, "wb") as f:
        async for chunk in request.stream():
            size += len(chunk)
            if size > 100 * 1024 * 1024:
                f.close()
                os.remove(path)
                return JSONResponse({"error": "file over 100 MB"}, status_code=413)
            f.write(chunk)
    try:
        probe(path)
    except Exception:
        os.remove(path)
        return JSONResponse({"error": "not a readable video"}, status_code=400)
    return JSONResponse({"clip": "uploads-broll/" + os.path.basename(path)})


def build_app(output_dir=None, **cfg):
    if output_dir:
        CFG["output_dir"] = output_dir
    CFG.update({k: v for k, v in cfg.items() if v is not None})
    return Starlette(routes=[
        Route("/", page),
        Route("/media/{path:path}", media),
        Route("/api/clips", api_clips),
        Route("/api/jobs", api_jobs, methods=["GET", "POST"]),
        Route("/api/jobs/{jid}", api_job),
        Route("/api/upload", api_upload, methods=["POST"]),
    ])


def unload_models():
    """Free GPU memory (e.g. before a long OpenShorts job)."""
    _models.update(sd=None, whisper=None)
    try:
        import gc, torch
        gc.collect()
        torch.cuda.empty_cache()
    except Exception:
        pass
''')
print("wrote", os.path.join(APP_DIR, 'broll.py'))

In [ ]:
# B-roll page
# Writes broll_ui.html into APP_DIR. Safe to re-run.
import os
APP_DIR = "/kaggle/working/studio_app"
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'broll_ui.html'), "w", encoding="utf-8") as f:
    f.write(r'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>B-roll · OpenShorts Studio</title>
<style>
:root{--bg:#f6f6f4;--panel:#fff;--text:#18181b;--muted:#6b6b73;--line:#e4e4e0;--accent:#e4572e;--accent-ink:#fff;--chip:#efefec;--ok:#15803d;--bad:#b91c1c}
@media (prefers-color-scheme:dark){:root{--bg:#111113;--panel:#1a1a1d;--text:#ececef;--muted:#9a9aa3;--line:#2a2a2f;--accent:#ff7a50;--accent-ink:#1a0c06;--chip:#25252a;--ok:#4ade80;--bad:#f87171}}
*{box-sizing:border-box}
body{margin:0;background:var(--bg);color:var(--text);font:15px/1.5 system-ui,-apple-system,"Segoe UI",Roboto,sans-serif}
header{display:flex;align-items:center;gap:16px;padding:12px 20px;border-bottom:1px solid var(--line);background:var(--panel);position:sticky;top:0;z-index:2}
header b{font-size:17px}header b span{color:var(--accent)}
header a{color:var(--muted);text-decoration:none;font-size:14px;margin-left:auto}
main{max-width:1200px;margin:0 auto;padding:18px 20px;display:grid;grid-template-columns:minmax(0,1.1fr) minmax(0,1fr);gap:18px}
@media (max-width:860px){main{grid-template-columns:1fr}}
.card{background:var(--panel);border:1px solid var(--line);border-radius:14px;padding:16px}
h2{font-size:15px;margin:0 0 10px}
.muted{color:var(--muted)}.small{font-size:13px}
.clips{display:grid;grid-template-columns:repeat(auto-fill,minmax(130px,1fr));gap:10px;max-height:62vh;overflow:auto;padding:2px}
.clip{border:2px solid transparent;border-radius:12px;background:var(--chip);padding:6px;cursor:pointer;text-align:left;color:inherit;font:inherit}
.clip.on{border-color:var(--accent)}
.clip video{width:100%;aspect-ratio:9/16;object-fit:cover;border-radius:8px;background:#000;display:block}
.clip .n{font-size:12px;margin-top:5px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.clip .tag{font-size:11px;color:var(--ok);font-weight:600}
label{display:block;font-size:13px;color:var(--muted);font-weight:600;margin:12px 0 5px}
input,select{width:100%;font:inherit;color:var(--text);background:var(--bg);border:1px solid var(--line);border-radius:9px;padding:8px 10px}
.row{display:flex;gap:10px;align-items:center;flex-wrap:wrap}
.row>div{flex:1;min-width:120px}
.row>.btns{flex:0 0 auto;min-width:0}
button.go{margin-top:14px;width:100%;font:inherit;font-weight:700;border:0;border-radius:10px;padding:12px;background:var(--accent);color:var(--accent-ink);cursor:pointer}
button.go:disabled{opacity:.5;cursor:default}
button.sec{font:inherit;font-size:13px;border:1px solid var(--line);background:var(--panel);color:var(--text);border-radius:8px;padding:6px 10px;cursor:pointer}
.bar{height:8px;background:var(--chip);border-radius:6px;overflow:hidden;margin:8px 0}
.bar div{height:100%;background:var(--accent);width:0;transition:width .4s}
.ins{display:grid;grid-template-columns:72px 1fr;gap:10px;align-items:start;margin:10px 0}
.ins img,.ins .ph{width:72px;aspect-ratio:9/16;object-fit:cover;border-radius:6px;background:var(--chip)}
.ins b{font-variant-numeric:tabular-nums}
video.result{width:100%;max-height:70vh;border-radius:10px;background:#000;margin-top:10px}
.err{color:var(--bad);white-space:pre-wrap;font-size:13px}
.empty{color:var(--muted);text-align:center;padding:30px 10px}
a.btn{display:inline-block;margin-top:8px;color:var(--accent);font-weight:600}
</style>
</head>
<body>
<header><b>OpenShorts <span>B-roll</span></b><span class="muted small" id="models"></span><a href="/">← Back to clips</a></header>
<main>
  <section class="card">
    <div class="row" style="justify-content:space-between"><h2 style="margin:0">1. Pick a clip</h2>
      <div class="row btns"><button class="sec" id="refresh">Refresh</button><button class="sec" id="upBtn">Upload mp4</button><input type="file" id="upFile" accept="video/mp4" hidden></div></div>
    <p class="muted small">Clips OpenShorts has made in this session. Uploads up to 100 MB.</p>
    <div class="clips" id="clips"><div class="empty">No clips yet. Make some on the main page first.</div></div>
  </section>

  <section class="card">
    <h2>2. Add b-roll</h2>
    <div class="muted small" id="picked">No clip selected.</div>
    <div class="row">
      <div><label for="count">Inserts</label><select id="count"><option>2</option><option selected>3</option><option>4</option><option>5</option></select></div>
      <div><label for="mode">Placement</label><select id="mode"><option value="upper" selected>Top 60% (captions stay visible)</option><option value="full">Full screen</option></select></div>
    </div>
    <label for="style">Visual style (optional)</label>
    <input id="style" placeholder="e.g. warm film photo, Kathmandu streets, 35mm">
    <button class="go" id="go" disabled>Add b-roll</button>
    <div id="job"></div>
  </section>
</main>
<script>
const $ = s => document.querySelector(s);
const esc = s => String(s ?? '').replace(/[&<>"']/g, c => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));
let picked = null, poll = null;
async function api(p, o) { const r = await fetch(p, o); const d = await r.json().catch(() => ({})); if (!r.ok) throw new Error(d.error || r.status); return d; }

async function loadClips() {
  const d = await api('api/clips');
  $('#models').textContent = `${d.model} · ${d.sd_model.split('/').pop()}`;
  $('#clips').innerHTML = d.clips.length ? d.clips.map(c => `<button class="clip${picked === c.clip ? ' on' : ''}" data-c="${esc(c.clip)}" title="${esc(c.title)}">
      <video src="${esc(c.url)}#t=0.5" muted preload="metadata"></video><div class="n">${esc(c.name)}</div>${c.broll_url ? '<div class="tag">has b-roll</div>' : ''}</button>`).join('')
    : '<div class="empty">No clips yet. Make some on the main page first, or upload an mp4.</div>';
}
$('#clips').onclick = e => { const b = e.target.closest('.clip'); if (!b) return; picked = b.dataset.c;
  document.querySelectorAll('.clip').forEach(x => x.classList.toggle('on', x === b)); $('#picked').textContent = picked; $('#go').disabled = false; };
$('#refresh').onclick = () => loadClips().catch(e => alertBox(e.message));
$('#upBtn').onclick = () => $('#upFile').click();
$('#upFile').onchange = async () => {
  const f = $('#upFile').files[0]; if (!f) return;
  if (f.size > 100 * 1024 * 1024) return alertBox('That file is over 100 MB. Add it as a Kaggle dataset instead.');
  $('#upBtn').textContent = 'Uploading…';
  try { const d = await api('api/upload?name=' + encodeURIComponent(f.name), {method: 'POST', body: f}); picked = d.clip; await loadClips();
        $('#picked').textContent = picked; $('#go').disabled = false; }
  catch (e) { alertBox(e.message); } finally { $('#upBtn').textContent = 'Upload mp4'; $('#upFile').value = ''; }
};
function alertBox(m) { $('#job').innerHTML = `<p class="err">${esc(m)}</p>`; }
$('#go').onclick = async () => {
  $('#go').disabled = true;
  try { const j = await api('api/jobs', {method: 'POST', headers: {'Content-Type': 'application/json'},
          body: JSON.stringify({clip: picked, count: +$('#count').value, mode: $('#mode').value, style: $('#style').value})});
        watch(j.id); }
  catch (e) { alertBox(e.message); $('#go').disabled = false; }
};
function render(j) {
  const ins = (j.inserts || []).map(it => `<div class="ins">${it.image_url ? `<img src="${esc(it.image_url)}" alt="">` : '<div class="ph"></div>'}
      <div><b>${it.start.toFixed(1)}s – ${(it.start + it.duration).toFixed(1)}s</b><div class="small muted">${esc(it.prompt)}</div></div></div>`).join('');
  $('#job').innerHTML = `<div style="margin-top:16px"><div class="row" style="justify-content:space-between"><b>${esc(j.step)}</b><span class="small muted">${Math.round(j.progress * 100)}%</span></div>
    <div class="bar"><div style="width:${j.progress * 100}%"></div></div>${j.error ? `<p class="err">${esc(j.error)}</p>` : ''}${ins}
    ${j.result_url ? `<video class="result" src="${esc(j.result_url)}" controls playsinline></video><a class="btn" href="${esc(j.result_url)}" download>Download clip</a>` : ''}</div>`;
}
function watch(id) {
  clearInterval(poll);
  const tick = async () => { try { const j = await api('api/jobs/' + id); render(j);
      if (j.state === 'done' || j.state === 'error') { clearInterval(poll); $('#go').disabled = !picked; loadClips(); } } catch (e) { alertBox(e.message); } };
  tick(); poll = setInterval(tick, 2000);
}
loadClips().catch(e => alertBox(e.message));
</script>
</body>
</html>
''')
print("wrote", os.path.join(APP_DIR, 'broll_ui.html'))

In [ ]:
# 4. Start Ollama (GPU 1, or GPU 0 if there is only one) and pull the model
import os, subprocess, time, requests, torch

N_GPU = torch.cuda.device_count()
LLM_GPU = "1" if N_GPU > 1 else "0"
if N_GPU < 2:
    print("Only %d GPU: everything shares it. Pick 'GPU T4 x2' for a smoother run." % N_GPU)

if "ollama_proc" in globals() and ollama_proc.poll() is None:
    ollama_proc.terminate(); ollama_proc.wait(10)
ollama_env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", CUDA_VISIBLE_DEVICES=LLM_GPU,
                  OLLAMA_CONTEXT_LENGTH="16384", OLLAMA_KEEP_ALIVE="30m")
ollama_proc = subprocess.Popen(["ollama", "serve"], env=ollama_env, stdout=open("/kaggle/working/ollama.log", "a"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=2); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start - see /kaggle/working/ollama.log")
subprocess.run(["ollama", "pull", LLM_MODEL], env=ollama_env, check=True)
print("Ollama ready with", LLM_MODEL, "on GPU", LLM_GPU)

In [ ]:
# 5. Start the OpenShorts backend on 127.0.0.1:8000 (GPU 0)
import os, subprocess, time, requests

def kaggle_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name) or None
    except Exception:
        return None

if "backend_proc" in globals() and backend_proc.poll() is None:
    backend_proc.terminate(); backend_proc.wait(15)
backend_env = dict(os.environ,
    CUDA_VISIBLE_DEVICES="0", PYTHONUNBUFFERED="1",
    LLM_BASE_URL="http://127.0.0.1:11434/v1", LLM_MODEL=LLM_MODEL, LLM_TIMEOUT="900",
    WHISPER_DEVICE="cuda", WHISPER_COMPUTE="float16", WHISPER_MODEL=WHISPER_MODEL,
    FFMPEG_ENCODER="auto", MAX_CONCURRENT_JOBS="1")
cookies = kaggle_secret("OPENSHORTS_YT_COOKIES")
if cookies:
    backend_env["YOUTUBE_COOKIES"] = cookies
backend_log = open("/kaggle/working/openshorts-backend.log", "a")
backend_proc = subprocess.Popen(["python", "-m", "uvicorn", "app:app", "--host", "127.0.0.1", "--port", "8000",
                                 "--proxy-headers", "--forwarded-allow-ips", "*"],
                                cwd=REPO_DIR, env=backend_env, stdout=backend_log, stderr=subprocess.STDOUT)
for _ in range(180):
    if backend_proc.poll() is not None:
        break
    try:
        if requests.get("http://127.0.0.1:8000/health/ready", timeout=2).ok:
            print("OpenShorts backend ready:", requests.get("http://127.0.0.1:8000/api/config", timeout=5).json().get("localLlm"))
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print("Backend is slow to start; check below.")
if backend_proc.poll() is not None:
    print("".join(open("/kaggle/working/openshorts-backend.log").readlines()[-40:]))
    raise RuntimeError("OpenShorts backend exited - log above")

In [ ]:
# 6. Start the gateway + B-roll page on port 7860
import sys, secrets, importlib, requests, torch
if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)
need = ["gateway.py", "broll.py", "broll_ui.html"]
missing = [f for f in need if not os.path.exists(os.path.join(APP_DIR, f))]
if missing:
    raise RuntimeError("Run the app-file cells above first (missing: %s)" % ", ".join(missing))

PASSWORD = kaggle_secret("OPENSHORTS_UI_PASSWORD")
if not PASSWORD:
    PASSWORD = secrets.token_urlsafe(9)
    print("No OPENSHORTS_UI_PASSWORD secret - this session's password:", PASSWORD)

import gateway, broll
if "gw" in globals():
    gateway.stop()
gateway = importlib.reload(gateway); broll = importlib.reload(broll)
sd_gpu = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
broll_app = broll.build_app(output_dir=os.path.join(REPO_DIR, "output"), llm_model=LLM_MODEL, sd_model=SD_MODEL, sd_steps=SD_STEPS,
                            sd_device=sd_gpu, whisper_index=int(sd_gpu[-1]), unload_after=torch.cuda.device_count() < 2)
gw = gateway.start(port=PORT, password=PASSWORD, dist=os.path.join(REPO_DIR, "dashboard/dist"), broll_app=broll_app)
time.sleep(2)
for host in ("127.0.0.1", "[::1]"):
    try:
        print(host, "->", requests.get("http://%s:%d/healthz" % (host, PORT), timeout=3).json())
    except Exception as e:
        print(host, "-> not reachable:", type(e).__name__)
print("Login: any username + the password. Agents/MCP: header 'Authorization: Bearer <password>'.")

In [ ]:
# 7. Start this notebook's Cloudflare Tunnel
TOKEN = kaggle_secret("OPENSHORTS_TUNNEL_TOKEN")
assert TOKEN, "Add the tunnel token as a Kaggle secret named OPENSHORTS_TUNNEL_TOKEN and attach it"
if "tunnel_proc" in globals() and tunnel_proc.poll() is None:
    tunnel_proc.terminate()
tunnel_proc = subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--no-autoupdate", "run"],
                               stdout=open("/kaggle/working/cloudflared.log", "a"), stderr=subprocess.STDOUT,
                               env={**os.environ, "TUNNEL_TOKEN": TOKEN})
for _ in range(30):
    time.sleep(1)
    if "Registered tunnel connection" in open("/kaggle/working/cloudflared.log").read():
        print("Tunnel connected. Open your hostname: dashboard at /, b-roll at /broll, MCP at /mcp")
        break
else:
    print("".join(open("/kaggle/working/cloudflared.log").readlines()[-15:]))

In [ ]:
# 8. Keep-alive monitor. Leave it running; interrupt to stop watching (everything keeps running).
import datetime
while True:
    def up(url):
        try:
            return "up" if requests.get(url, timeout=3).ok else "DOWN"
        except Exception:
            return "DOWN"
    gpu = ", ".join("%d: %.1f/%.0f GB" % (i, (torch.cuda.mem_get_info(i)[1] - torch.cuda.mem_get_info(i)[0]) / 1e9,
                                         torch.cuda.mem_get_info(i)[1] / 1e9) for i in range(torch.cuda.device_count()))
    running = [j["step"] for j in broll.JOBS.values() if j["state"] in ("queued", "running")]
    print("%s gateway=%s backend=%s ollama=%s tunnel=%s | GPU %s%s" % (
        datetime.datetime.now().strftime("%H:%M:%S"), up("http://127.0.0.1:%d/healthz" % PORT),
        up("http://127.0.0.1:8000/health"), up("http://127.0.0.1:11434/api/tags"),
        "up" if tunnel_proc.poll() is None else "DOWN", gpu, (" | b-roll: " + running[0]) if running else ""), flush=True)
    time.sleep(60)